<a href="https://colab.research.google.com/github/Priyank0302/ML_Project/blob/main/notebooks/task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install PySpark in Colab
Run this in the first cell. Google Colab doesn't come with PySpark pre-installed.

In [1]:
!pip install pyspark

### Start a Spark Session
This is your entry point to everything PySpark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(spark)

### Load the Dataset from Hugging Face
The dataset is stored as 14 CSV files (1.5M rows each) on Hugging Face. Load them directly via URL.

In [3]:
import requests
from pyspark import SparkFiles

# Base URL for the dataset files on Hugging Face
BASE_URL = "https://huggingface.co/datasets/CiferAI/Cifer-Fraud-Detection-Dataset-AF/resolve/main/"

# Generate all 14 file URLs
file_urls = [
    f"{BASE_URL}Cifer-Fraud-Detection-Dataset-AF-part-{i}-14.csv"
    for i in range(1, 15)
]

# Add each file to the Spark context
for url in file_urls[:7]:
    spark.sparkContext.addFile(url)

# Read all CSV files into a single Spark DataFrame
import os
root_dir = SparkFiles.getRootDirectory()

df = spark.read.csv(
    root_dir + "/*.csv",
    header=True,
    inferSchema=True
)

In [4]:
# Print Dataset Info
df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



In [5]:
# Row and column count ("shape")

print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")

Rows: 10500000
Columns: 11


In [6]:
# Column names list

print(df.columns)

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [7]:
# Show first 5 rows (like pandas head())
df.show(5)

# Show first 10 rows with full column width
df.show(10, truncate=False)

+----+--------+---------+-------------+-------------+--------------+-------------+--------------+--------------+-------+--------------+
|step|    type|   amount|     nameOrig|oldbalanceOrg|newbalanceOrig|     nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+---------+-------------+-------------+--------------+-------------+--------------+--------------+-------+--------------+
| 371|CASH_OUT|367336.05|sdv-pii-r8zd6|   4514816.83|    2108392.86|sdv-pii-q6998|    1265486.06|    2454140.46|      0|             0|
| 368|TRANSFER|   238.63|sdv-pii-xq6z3|    430944.71|     1865444.6|sdv-pii-n2ql8|     107927.46|       2021.16|      0|             0|
| 141|CASH_OUT|   254.93|sdv-pii-805w0|    839593.53|    8008353.88|sdv-pii-yo0z6|     773352.22|         20.79|      0|             0|
| 191| CASH_IN|501547.39|sdv-pii-279tw|      41226.4|      28633.52|sdv-pii-9zlyl|    6825363.55| 1.644207824E7|      0|             0|
| 169|TRANSFER|  71832.0|sdv-pii-ksz58|     2486

In [8]:

# Statistical summary of numeric columns (like pandas describe())
df.describe().show()

+-------+------------------+--------+------------------+-------------+------------------+------------------+-------------+------------------+--------------------+--------------------+--------------------+
|summary|              step|    type|            amount|     nameOrig|     oldbalanceOrg|    newbalanceOrig|     nameDest|    oldbalanceDest|      newbalanceDest|             isFraud|      isFlaggedFraud|
+-------+------------------+--------+------------------+-------------+------------------+------------------+-------------+------------------+--------------------+--------------------+--------------------+
|  count|          10500000|10500000|          10500000|     10500000|          10500000|          10500000|     10500000|          10500000|            10500000|            10500000|            10500000|
|   mean|238.88391304761905|    NULL|179870.08930465687|         NULL|1337218.6290398182|2088560.4657765916|         NULL|1939048.9910791786|  5592477.9689257415|0.0013099047619047

In [9]:
# Value counts for transaction type
df.groupBy("type").count().orderBy("count", ascending=False).show()

# Check fraud label distribution
df.groupBy("isFraud").count().show()

# Check flagged fraud distribution
df.groupBy("isFlaggedFraud").count().show()

+--------+-------+
|    type|  count|
+--------+-------+
|CASH_OUT|3695491|
| PAYMENT|3549092|
| CASH_IN|2307021|
|TRANSFER| 880073|
|   DEBIT|  68323|
+--------+-------+

+-------+--------+
|isFraud|   count|
+-------+--------+
|      1|   13754|
|      0|10486246|
+-------+--------+

+--------------+--------+
|isFlaggedFraud|   count|
+--------------+--------+
|             1|      28|
|             0|10499972|
+--------------+--------+



### Check for Nulls

In [11]:
from pyspark.sql.functions import col, isnan, when, count

null_checks = []
for c, dtype in df.dtypes:
    if dtype in ["int", "double", "float", "long", "short", "decimal"]:
        # Apply isnan only to numeric columns
        null_checks.append(count(when(col(c).isNull() | isnan(col(c)), c)).alias(c))
    else:
        # For non-numeric columns, only check for null
        null_checks.append(count(when(col(c).isNull(), c)).alias(c))

null_counts = df.select(null_checks)
null_counts.show()

+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|   0|   0|     0|       0|            0|             0|       0|             0|             0|      0|             0|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+

